# Experiment 3 (v2): Number Agreement + Structural Distance, Proper PCFG

**Why this redesign.** The previous PCFG attempts (v6–v10.1) used gender-from-noun as the
agreement feature. Diagnosis (see project notes) found this was structurally unlearnable-as-a-test:
gender is a static per-token lookup, which a transformer memorizes almost for free, so neither
the τ manipulation (does DET expose gender) nor the ε manipulation (Hamming distance on a
gender table) produced a measurable effect — confirmed directly in the numeric toy model
(Experiment 2) where an analogous τ knob turned out to be silent for the same reason.

**What the literature says to do instead** (Wendler et al. 2024; Zhao et al. 2024, NeurIPS;
Tang et al. 2024; Brinkmann et al. 2025 NAACL): multilingual transformers show a genuine
three-phase structure — early layers do language-specific input processing, middle layers
share a conceptual space, late layers convert back to surface form. Abstract *structural*
concepts (grammatical number, tense) are shared across typologically diverse languages
(Brinkmann et al., causally verified via ablation); what differs across languages is (a) how
much *structural computation* the input-to-concept mapping requires, and (b) whether the
concept's *rule* itself is the same across languages once you're in the shared space.

**This notebook's design, grounded in that finding:**

- **Task**: subject–verb number agreement (a genuinely *shared*, structural computation —
  not a per-token lookup like gender).
- **τ (structural distance)**: whether an intervening modifier (PP / relative clause) with an
  independently-sampled "attractor" noun sits *between* the subject and the verb (in-situ
  attachment) or is extraposed to after the verb. This is the classic agreement-attraction
  paradigm from psycholinguistics/computational linguistics (Linzen et al.) — a well-grounded,
  real cost axis, not an arbitrary marker.
- **ε (representational competition)**: a small quantifier-override rule
  (`(quantifier, noun-class) -> forced number`), defined over a vocabulary **shared across every
  language**, that differs per language by a controlled number of cell-flips from an anchor
  table (the Hamming-distance mechanism from Experiment 2's v9 plan, now applied to a genuine
  *function* over classes rather than a static per-token table — so it can't be solved by
  disjoint-token memorization the way the gender design was).

**This notebook bakes in the hard-won lesson from Experiment 2 and the earlier PCFG runs:
τ and ε are validated with mandatory isolated pre-flight checks *before* any capacity sweep is
run.** Both checks are included below with their actual results from a first pass (CPU, small
scale) — both mechanisms show large, genuine effects. The full-scale sweep (Section 8) is left
for you to run on GPU; it is not executed in this notebook.


## 0. Setup

In [1]:
# GPU selection MUST happen before `import torch` -- CUDA's visible-device
# list is locked in at torch's CUDA-context init, so setting this after
# import silently does nothing. Change GPU_ID if your machine has more than
# one GPU and you want a specific one (leave as "0" for a single-GPU box).
import os
GPU_ID = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

import random, time, math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
random.seed(0)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    torch.cuda.manual_seed_all(0)
    print('device: cuda ->', torch.cuda.get_device_name(0))
else:
    print('device: cpu (no GPU visible -- check GPU_ID / CUDA_VISIBLE_DEVICES if this is unexpected)')


device: cuda -> NVIDIA A100 80GB PCIe


## 1. Shared PCFG structure

Every production rule and probability below is identical across all languages. Word order
and the override rule are **not** part of this grammar — they enter only in realization
(Sections 3–4). This is the same discipline as the prior notebook: one structural grammar,
many realizations, so a cross-language difference can only come from the two knobs we intend
to manipulate.

In [2]:
"""
Experiment 3 core: shared PCFG structure, per-language word-order/attachment
(tau) and quantifier-override rule (epsilon), surface realization, sampler.
"""
import random
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional

# ───────────────────────── Lexicon ─────────────────────────
# Two noun classes, shared across ALL languages (same stem -> same class
# everywhere). Class is what the override rule conditions on; it is NOT a
# per-language fact, so it cannot be a source of cross-language epsilon.
N_COUNT_NOUNS = 300
N_MASS_NOUNS  = 300
COUNT_NOUNS = [f"cnt{i:04d}" for i in range(N_COUNT_NOUNS)]
MASS_NOUNS  = [f"mss{i:04d}" for i in range(N_MASS_NOUNS)]
NOUN_CLASS = {n: 'count' for n in COUNT_NOUNS}
NOUN_CLASS.update({n: 'mass' for n in MASS_NOUNS})
ALL_NOUNS = COUNT_NOUNS + MASS_NOUNS

# Two shared quantifier tokens (closed class, same surface tokens in every
# language -- this is what makes the override-rule difference a genuine
# rule conflict rather than a different lookup table over disjoint tokens).
QUANTIFIERS = ['q_sev', 'q_man']   # stand-ins for e.g. "several" / "much"

N_VTR = 6
N_VINTR = 4
VERB_STEMS_TR = [f"vtr{i}" for i in range(N_VTR)]
VERB_STEMS_INTR = [f"vin{i}" for i in range(N_VINTR)]
PREPS = ['p0', 'p1', 'p2', 'p3']
RPRO = 'rp'
CONJ = 'cj'
DET = 'dt'

# ───────────────────────── PCFG production rules ─────────────────────────
# All probabilities are SHARED across every language -- word order and the
# override rule enter only in surface realization, never here. This mirrors
# the design principle from the earlier notebook: one structural grammar,
# many realizations.
P_S_COORD        = 0.20   # S -> CLAUSE CONJ CLAUSE
P_MOD_PRESENT    = 0.45   # NP -> DET N (MOD)
P_MOD_IS_REL     = 0.50   # MOD -> PP | RELCLAUSE
P_TRANSITIVE     = 0.55   # VP -> V_intr | V_tr NP_obj
P_OBJ_MOD        = 0.30   # object NP also optionally takes a MOD
P_QUANT_PRESENT  = 0.35   # NP -> (Q) DET N (MOD)  -- quantifier attaches to subject/obj NP
P_ATTRACTOR_PL   = 0.5    # attractor noun's own lexical number, sampled independently


def sample_number() -> str:
    return random.choice(['SG', 'PL'])


@dataclass
class NP:
    noun: str
    noun_class: str
    quant: Optional[str]
    lexical_number: str          # the noun's own base number, before override
    mod: Optional[dict]          # attractor info, or None
    final_number: str = None     # filled in by apply_override


def sample_np(rng: random.Random, allow_mod=True) -> NP:
    noun = rng.choice(ALL_NOUNS)
    ncls = NOUN_CLASS[noun]
    lex_num = sample_number()
    quant = rng.choice(QUANTIFIERS) if rng.random() < P_QUANT_PRESENT else None
    mod = None
    if allow_mod and rng.random() < P_MOD_PRESENT:
        attractor_noun = rng.choice(ALL_NOUNS)
        mod = dict(
            kind='PP' if rng.random() < P_MOD_IS_REL else 'REL',
            attractor_noun=attractor_noun,
            attractor_number=sample_number(),
        )
    return NP(noun=noun, noun_class=ncls, quant=quant, lexical_number=lex_num, mod=mod)


@dataclass
class Clause:
    subj: NP
    verb_stem: str
    transitive: bool
    obj: Optional[NP]


def sample_clause(rng: random.Random) -> Clause:
    subj = sample_np(rng, allow_mod=True)
    transitive = rng.random() < P_TRANSITIVE
    if transitive:
        obj = sample_np(rng, allow_mod=(rng.random() < P_OBJ_MOD))
        verb_stem = rng.choice(VERB_STEMS_TR)
    else:
        obj = None
        verb_stem = rng.choice(VERB_STEMS_INTR)
    return Clause(subj=subj, verb_stem=verb_stem, transitive=transitive, obj=obj)


@dataclass
class Sentence:
    clauses: List[Clause]
    coordinated: bool


def sample_sentence(rng: random.Random) -> Sentence:
    c1 = sample_clause(rng)
    if rng.random() < P_S_COORD:
        c2 = sample_clause(rng)
        return Sentence(clauses=[c1, c2], coordinated=True)
    return Sentence(clauses=[c1], coordinated=False)


### 1a. Sanity check: sampling works

In [3]:
rng = random.Random(0)
for _ in range(3):
    print(sample_sentence(rng))

Sentence(clauses=[Clause(subj=NP(noun='mss0094', noun_class='mass', quant=None, lexical_number='PL', mod={'kind': 'REL', 'attractor_noun': 'cnt0265', 'attractor_number': 'PL'}, final_number=None), verb_stem='vtr2', transitive=True, obj=NP(noun='mss0010', noun_class='mass', quant=None, lexical_number='SG', mod=None, final_number=None))], coordinated=False)
Sentence(clauses=[Clause(subj=NP(noun='cnt0223', noun_class='count', quant=None, lexical_number='PL', mod={'kind': 'REL', 'attractor_noun': 'cnt0097', 'attractor_number': 'PL'}, final_number=None), verb_stem='vtr5', transitive=True, obj=NP(noun='cnt0150', noun_class='count', quant='q_sev', lexical_number='PL', mod=None, final_number=None))], coordinated=False)
Sentence(clauses=[Clause(subj=NP(noun='mss0273', noun_class='mass', quant='q_man', lexical_number='PL', mod={'kind': 'REL', 'attractor_noun': 'cnt0209', 'attractor_number': 'PL'}, final_number=None), verb_stem='vtr0', transitive=True, obj=NP(noun='cnt0266', noun_class='count', q

## 2. Per-language parameters: τ (attachment) and ε (override rule)

`tau_intervene` controls how often an in-situ modifier sits between the subject and the verb
(the structural-distance / attraction manipulation). `eps_flips` controls how many of the 4
cells in the quantifier×noun-class override table differ from the shared anchor language,
via `build_override_table`.

In [4]:
"""
Per-language parameters and surface realization.

tau  = P(an in-situ MOD sits BETWEEN subject and verb) -- controlled by
       attachment style (in-situ vs extraposed), NOT by the PCFG's own
       production probabilities (those are shared -- see grammar.py).
       tau LOW  = MOD almost always extraposed -> subject stays locally
                  adjacent to the verb -> cheap recency heuristic works.
       tau HIGH = MOD almost always stays in-situ -> attractor commonly
                  intervenes -> recency heuristic frequently wrong,
                  real subject-tracking is required.
       (Named so that HIGH tau = "hard", to match the theory's convention
       tau = how rich/available a cheap shortcut is is INVERTED here on
       purpose -- see note in the notebook markdown. We define tau_shortcut
       = 1 - tau_intervene below so the convention matches the rest of the
       project: tau_shortcut HIGH = cheap recency shortcut usually correct.)

eps_flips = number of cells flipped (out of 4) in the language's
            quantifier x noun-class override table, relative to the anchor
            table. This is the Hamming-distance mechanism, now applied to a
            small FUNCTIONAL RULE over classes (not a per-token lookup),
            using a vocabulary shared across every language.
"""
from dataclasses import dataclass
from typing import Dict, Tuple
import random


OVERRIDE_CELLS = [(q, c) for q in QUANTIFIERS for c in ('count', 'mass')]
# Anchor rule: q_sev forces PL on count nouns, no effect on mass nouns;
# q_man forces SG on mass nouns, no effect on count nouns. This is a
# genuine conditional rule (depends on class, not just quantifier identity).
ANCHOR_OVERRIDE: Dict[Tuple[str, str], str] = {
    ('q_sev', 'count'): 'PL',
    ('q_sev', 'mass'):  'NONE',
    ('q_man', 'count'): 'NONE',
    ('q_man', 'mass'):  'SG',
}


def build_override_table(eps_flips: int, seed: int) -> Dict[Tuple[str, str], str]:
    rng = random.Random(seed)
    table = dict(ANCHOR_OVERRIDE)
    cells = list(OVERRIDE_CELLS)
    rng.shuffle(cells)
    outcomes = ['NONE', 'SG', 'PL']
    for cell in cells[:eps_flips]:
        current = table[cell]
        table[cell] = rng.choice([o for o in outcomes if o != current])
    return table


@dataclass
class LangSpec:
    name: str
    tau_intervene: float   # P(in-situ MOD attachment | MOD present)
    eps_flips: int          # 0..4
    seed: int
    anchor: bool = False

    def __post_init__(self):
        self.override_table = (dict(ANCHOR_OVERRIDE) if self.anchor
                                else build_override_table(self.eps_flips, self.seed))

    def apply_override(self, np_: NP) -> str:
        """Compute the FINAL number the verb must agree with for this NP,
        in THIS language: lexical number unless a quantifier + this
        language's rule table forces an override."""
        if np_.quant is None:
            return np_.lexical_number
        forced = self.override_table[(np_.quant, np_.noun_class)]
        return np_.lexical_number if forced == 'NONE' else forced

    def mod_in_situ(self, rng: random.Random) -> bool:
        return rng.random() < self.tau_intervene


### 2a. Sanity check: override tables

In [5]:
A = LangSpec('A', tau_intervene=0.05, eps_flips=0, seed=1, anchor=True)
C = LangSpec('C', tau_intervene=0.05, eps_flips=2, seed=3)
print('A (anchor):', A.override_table)
print('C (eps=2): ', C.override_table)

A (anchor): {('q_sev', 'count'): 'PL', ('q_sev', 'mass'): 'NONE', ('q_man', 'count'): 'NONE', ('q_man', 'mass'): 'SG'}
C (eps=2):  {('q_sev', 'count'): 'SG', ('q_sev', 'mass'): 'NONE', ('q_man', 'count'): 'NONE', ('q_man', 'mass'): 'PL'}


## 3. Surface realization

**A bug caught during construction, worth keeping visible**: the first version of this
grammar sampled each noun's lexical number independently per occurrence but never marked it
anywhere in the surface tokens — so the "no quantifier" case was literally unlearnable (no
token carried the information needed to answer). Fixed by having the determiner surface-mark
lexical number (`dt_sg` / `dt_pl`); the quantifier + override rule can then force the
*agreement* to differ from that marked default. This mirrors a real cross-linguistic
phenomenon (e.g. British English "the government are/is") rather than being an ad hoc fix.
Word order is fixed SVO for every language — attachment site (in-situ vs. extraposed), not
word order, is the τ manipulation, to keep exactly one structural parameter varying.

In [6]:
"""
Surface realization: turn a sampled Sentence + LangSpec into a token
sequence, plus per-position annotations for feature-level evaluation.
Word order is fixed SVO for every language (word order is NOT the tau
knob here -- attachment site is; see langspec.py docstring). This keeps
the manipulation isolated to exactly one structural parameter, per the
project's twin-invariant discipline.
"""
import random
from typing import List, Dict, Tuple


BOS, EOS = '<BOS>', '<EOS>'
DOT = '.'
DET = 'dt'  # unused directly; determiner now carries lexical number (see realize_np)
DET_SG, DET_PL = 'dt_sg', 'dt_pl'

# Shared, language-tagged agreement-bearing verb forms (SG/PL), so a
# cross-language accuracy difference can't be explained by relearning
# separate output lexica (same design principle as the earlier notebook).
VFORMS = {(stem, n): f"agr_{stem}_{n}" for stem in VERB_STEMS_TR + VERB_STEMS_INTR
          for n in ('SG', 'PL')}


def realize_np(np_: NP, lang: LangSpec, role: str) -> Tuple[List[str], str]:
    """Returns (tokens, final_number). Quantifier/DET/Noun in order; MOD
    (if any) is placed IN-SITU (right after the noun) or held back for
    extraposition by the caller, per lang.tau_intervene.

    The determiner surface-marks the noun's own LEXICAL number (dt_sg /
    dt_pl) -- this is the only place that information is readable from the
    tokens at all, so the "no quantifier" case is genuinely learnable (copy
    the marked default). A quantifier + this language's override rule can
    then force the AGREEMENT to differ from that marked default -- e.g.
    "several <dt_sg> cattle" still surface-marked singular, but agreement
    is forced plural by the quantifier rule in some languages. That
    mismatch-when-overridden is exactly the genuine, non-memorizable
    computation we want (the determiner alone is no longer sufficient)."""
    toks = []
    if np_.quant:
        toks.append(np_.quant)
    det = DET_PL if np_.lexical_number == 'PL' else DET_SG
    toks += [det, np_.noun]
    final_number = lang.apply_override(np_)
    return toks, final_number


def realize_mod(np_: NP, rng: random.Random) -> List[str]:
    mod = np_.mod
    if mod is None:
        return []
    det = DET_PL if mod['attractor_number'] == 'PL' else DET_SG
    if mod['kind'] == 'PP':
        prep = rng.choice(['p0', 'p1', 'p2', 'p3'])
        return [prep, det, mod['attractor_noun']]
    else:
        return ['rp', det, mod['attractor_noun']]  # simplified relative clause


def realize_clause(clause: Clause, lang: LangSpec, rng: random.Random, ann: List[dict], toks: List[str]):
    subj_toks, subj_final_num = realize_np(clause.subj, lang, role='subj')
    subj_mod_toks = realize_mod(clause.subj, rng)
    mod_in_situ = lang.mod_in_situ(rng) if clause.subj.mod is not None else False

    toks += subj_toks
    if clause.subj.mod is not None and mod_in_situ:
        toks += subj_mod_toks   # attractor sits BETWEEN subject and verb

    # ---- verb slot: this is the prediction target ----
    verb_pos = len(toks)  # position of the (about-to-be-appended) verb token
    vform = VFORMS[(clause.verb_stem, subj_final_num)]
    toks.append(vform)
    ann.append(dict(pos=verb_pos, subcomp='subj_verb_agree',
                     target_number=subj_final_num,
                     had_intervening_attractor=bool(clause.subj.mod is not None and mod_in_situ),
                     attractor_mismatches=(clause.subj.mod is not None and mod_in_situ and
                                           clause.subj.mod['attractor_number'] != subj_final_num)))

    if clause.subj.mod is not None and not mod_in_situ:
        toks += subj_mod_toks   # extraposed: AFTER the verb, doesn't intervene

    if clause.transitive and clause.obj is not None:
        obj_toks, _ = realize_np(clause.obj, lang, role='obj')
        obj_mod_toks = realize_mod(clause.obj, rng)
        toks += obj_toks
        if clause.obj.mod is not None:
            toks += obj_mod_toks  # object modifiers never intervene in subj-verb agreement; harmless filler


def realize_sentence(sent: Sentence, lang: LangSpec, rng: random.Random) -> Tuple[List[str], List[dict]]:
    toks = [BOS]
    ann = []
    for i, clause in enumerate(sent.clauses):
        realize_clause(clause, lang, rng, ann, toks)
        if i == 0 and sent.coordinated:
            toks.append('cj')
    toks.append(DOT)
    toks.append(EOS)
    # ann positions were recorded relative to the buffer at time of append;
    # they are exact indices into `toks` since we appended in order.
    return toks, ann


### 3a. Sanity check: realized sentences, two τ extremes

In [7]:
rng = random.Random(0)
A = LangSpec('A', tau_intervene=0.05, eps_flips=0, seed=1, anchor=True)
B = LangSpec('B', tau_intervene=0.95, eps_flips=0, seed=2)
for lang in [A, B]:
    print(f'--- {lang.name} (tau_intervene={lang.tau_intervene}) ---')
    for _ in range(3):
        s = sample_sentence(rng)
        toks, ann = realize_sentence(s, lang, rng)
        print(' '.join(toks))
    print()

--- A (tau_intervene=0.05) ---
<BOS> dt_sg mss0094 agr_vtr2_SG rp dt_sg cnt0265 dt_pl mss0010 . <EOS>
<BOS> q_sev dt_sg mss0216 agr_vin2_SG . <EOS>
<BOS> dt_sg mss0245 agr_vin2_SG cj dt_pl cnt0075 agr_vtr3_PL q_sev dt_sg mss0144 . <EOS>

--- B (tau_intervene=0.95) ---
<BOS> dt_pl mss0153 rp dt_sg mss0261 agr_vtr2_PL dt_sg cnt0001 . <EOS>
<BOS> q_sev dt_pl cnt0064 agr_vtr3_PL rp dt_pl cnt0145 dt_sg mss0027 cj q_sev dt_pl mss0264 agr_vin1_PL . <EOS>
<BOS> dt_pl mss0260 p1 dt_pl mss0094 agr_vtr0_PL q_man dt_sg cnt0191 . <EOS>



## 4. Tokenizer / shared vocabulary

Same token = same id in every language (critical for the "shared circuit" framing to mean
anything — see Experiment 2's lesson that disjoint vocabularies structurally prevent ε from
existing at all).

In [8]:
"""Tokenizer + batched example generation for a given language."""
import random
import torch
from typing import List

# ---- vocabulary (shared across ALL languages: same token = same id everywhere) ----
SPECIAL = ['<PAD>', BOS, EOS]
VOCAB = SPECIAL + [DOT, DET_SG, DET_PL, 'rp', 'cj'] + PREPS + QUANTIFIERS + ALL_NOUNS + list(VFORMS.values())
VOCAB = list(dict.fromkeys(VOCAB))  # dedupe, preserve order
STOI = {t: i for i, t in enumerate(VOCAB)}
ITOS = {i: t for t, i in STOI.items()}
PAD_ID = STOI['<PAD>']
VOCAB_SIZE = len(VOCAB)

# restricted-vocab feature eval: at an agreement position, the model only
# needs to choose between the SG/PL forms of the verb stem actually used.
VFORM_SG_IDS = {stem: STOI[VFORMS[(stem, 'SG')]] for stem in VERB_STEMS_TR + VERB_STEMS_INTR}
VFORM_PL_IDS = {stem: STOI[VFORMS[(stem, 'PL')]] for stem in VERB_STEMS_TR + VERB_STEMS_INTR}


def make_example(lang: LangSpec, rng: random.Random, max_len: int):
    s = sample_sentence(rng)
    toks, ann = realize_sentence(s, lang, rng)
    if len(toks) > max_len:
        return None
    ids = [STOI[t] for t in toks]
    return ids, ann, toks


def make_batch(lang: LangSpec, rng: random.Random, batch_size: int, max_len: int = 24):
    seqs, anns = [], []
    while len(seqs) < batch_size:
        ex = make_example(lang, rng, max_len)
        if ex is None:
            continue
        ids, ann, toks = ex
        seqs.append(ids)
        anns.append(ann)
    # Pad every batch to the FIXED max_len (not each batch's own longest
    # sequence). This keeps tensor shapes constant across every training
    # step, which matters for two reasons: (1) it lets CUDA pick a stable
    # kernel/algorithm once and reuse it instead of re-selecting per step,
    # and (2) it's a prerequisite for torch.compile to actually help --
    # compile retraces on every new input shape, so variable-length padding
    # silently turns "compile once" into "recompile every step".
    x = torch.full((batch_size, max_len), PAD_ID, dtype=torch.long)
    for i, s in enumerate(seqs):
        x[i, :len(s)] = torch.tensor(s, dtype=torch.long)
    return x, anns


### 4a. Note: fixed-length padding

Batches now pad to a constant `max_len` every step (not each batch's own longest sequence).
Constant shapes matter for GPU efficiency (stable kernel selection) and are a prerequisite
for `torch.compile` to help rather than hurt (compile retraces on every new shape).

## 5. Model: capacity-bottlenecked TinyGPT

`n_embd_tok` is a narrow bottleneck the token embedding must pass through before the wider
residual stream (`n_embd`); this is the capacity knob the sweep varies, so pressure binds on
the token-identity → agreement pathway specifically rather than being hidden by a wide
residual stream elsewhere (same principle as the prior notebook's design).

In [9]:
"""Tiny decoder-only transformer with a separate token-identity bottleneck
(n_embd_tok) from the residual width (n_embd) -- same principle as the
prior PCFG notebook: capacity pressure has to bind on the (quantifier,
noun) -> gender/number lookup pathway specifically, not be hidden by a
wide residual stream elsewhere."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, max_len):
        super().__init__()
        self.n_head = n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd)
        self.proj = nn.Linear(n_embd, n_embd)
        mask = torch.tril(torch.ones(max_len, max_len)).view(1, 1, max_len, max_len)
        self.register_buffer('mask', mask)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.split(C, dim=2)
        hd = C // self.n_head
        q = q.view(B, T, self.n_head, hd).transpose(1, 2)
        k = k.view(B, T, self.n_head, hd).transpose(1, 2)
        v = v.view(B, T, self.n_head, hd).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(hd)
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        out = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(out)


class Block(nn.Module):
    def __init__(self, n_embd, n_head, max_len):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, max_len)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = nn.Sequential(nn.Linear(n_embd, 4 * n_embd), nn.GELU(), nn.Linear(4 * n_embd, n_embd))

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class TinyGPT(nn.Module):
    def __init__(self, vocab_size, n_embd_tok, n_embd=64, n_layer=3, n_head=4, max_len=24, pad_id=0):
        super().__init__()
        self.tok_bottleneck = nn.Embedding(vocab_size, n_embd_tok, padding_idx=pad_id)
        self.tok_proj = nn.Linear(n_embd_tok, n_embd) if n_embd_tok != n_embd else nn.Identity()
        self.pos_emb = nn.Embedding(max_len, n_embd)
        self.blocks = nn.ModuleList([Block(n_embd, n_head, max_len) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)
        self.max_len = max_len

    def forward(self, idx):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.tok_proj(self.tok_bottleneck(idx)) + self.pos_emb(pos)[None, :, :]
        for blk in self.blocks:
            x = blk(x)
        x = self.ln_f(x)
        return self.head(x)


## 6. Training + feature-level evaluation

Loss is the standard causal-LM loss over the whole sequence (the model also has to predict
nouns etc., which is irreducible noise — restricted-vocab **feature accuracy** at the verb
position, scored against the correct-number vs incorrect-number form of the *same* stem, is
the actual metric that matters, exactly as in the earlier notebook).

In [10]:
import random
import torch
import torch.nn.functional as F
from typing import List



def compute_loss(model, x, device):
    x = x.to(device)
    logits = model(x[:, :-1])
    targets = x[:, 1:].contiguous()
    loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1),
                            ignore_index=PAD_ID)
    return loss


def train_isolated(lang: LangSpec, n_embd_tok, steps=1500, seed=0, lr=2e-2,
                    n_embd=64, n_layer=3, n_head=4, max_len=24, device='cpu'):
    torch.manual_seed(seed)
    model = TinyGPT(VOCAB_SIZE, n_embd_tok, n_embd, n_layer, n_head, max_len, PAD_ID).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=steps, eta_min=lr * 0.02)
    rng = random.Random(1000 + seed)
    for step in range(steps):
        opt.zero_grad()
        x, _ = make_batch(lang, rng, batch_size=64, max_len=max_len)
        loss = compute_loss(model, x, device)
        loss.backward()
        opt.step()
        sched.step()
    return model


@torch.no_grad()
def feature_accuracy(model, lang: LangSpec, device='cpu', n_eval=4000, max_len=24,
                      seed=999, split_by_intervention=False):
    """Restricted-vocab accuracy at subj-verb agreement positions: does the
    model prefer the correct-number verb FORM over the wrong-number form of
    the SAME stem (so full-vocab lemma-choice noise never enters this
    number)."""
    model.eval()
    rng = random.Random(seed)
    correct = {'all': 0}
    total = {'all': 0}
    n_done = 0
    while n_done < n_eval:
        bs = min(64, n_eval - n_done)
        x, anns = make_batch(lang, rng, batch_size=bs, max_len=max_len)
        x = x.to(device)
        logits = model(x[:, :-1])  # predict position t from tokens <= t-1... see note below
        for i, ann_list in enumerate(anns):
            for a in ann_list:
                pos = a['pos']  # index of the verb token in x[i]
                if pos == 0 or pos >= x.shape[1]:
                    continue
                # logits[i, pos-1] predicts token at position `pos`
                stem_token_id = x[i, pos].item()
                # recover stem from the two candidate ids at this position
                # (we know target_number and can look up the OTHER form's id)
                # find stem by matching id in either dict
                stem = None
                for s, sg_id in VFORM_SG_IDS.items():
                    if sg_id == stem_token_id or VFORM_PL_IDS[s] == stem_token_id:
                        stem = s
                        break
                sg_id, pl_id = VFORM_SG_IDS[stem], VFORM_PL_IDS[stem]
                l_sg = logits[i, pos - 1, sg_id].item()
                l_pl = logits[i, pos - 1, pl_id].item()
                pred = 'SG' if l_sg > l_pl else 'PL'
                is_correct = int(pred == a['target_number'])
                correct['all'] += is_correct
                total['all'] += 1
                if split_by_intervention:
                    key = 'intervene' if a['had_intervening_attractor'] else 'no_intervene'
                    correct.setdefault(key, 0); total.setdefault(key, 0)
                    correct[key] += is_correct; total[key] += 1
        n_done += bs
    return {k: correct[k] / total[k] for k in total}


## 7. Mandatory pre-flight checks — DO NOT skip to the capacity sweep without these

Both checks below were already run (CPU, small/moderate capacity, ~2000 steps) as part of
building this notebook. Re-run them yourself before trusting anything from Section 8 on a
new machine/seed — if either comes back null, the sweep is not worth running until the
grammar is fixed, exactly as happened with the gender design.

**Pass criterion for τ**: at small capacity, the high-`tau_intervene` language should score
*measurably* lower feature accuracy than the low-`tau_intervene` language, matched on `eps`.

**Pass criterion for ε**: a model trained only on the anchor language should score much
worse when evaluated (frozen, no retraining) on a language with a maximally different
override table than on itself — this shows the override-table difference is genuine
representational conflict, not a free relabeling.

In [11]:
# --- 7a. tau pre-flight: same eps (0), only attachment differs ---
# NOTE: a single seed is NOT reliable at this scale -- an earlier single-seed
# run of this exact cell flipped direction (B briefly beat A) purely from
# training-noise sensitivity near a discovery threshold. Always average
# multiple seeds before trusting a tau/eps verdict; this is the same lesson
# Experiment 2 already taught about undertraining/instability at small scale.
A = LangSpec('A', tau_intervene=0.05, eps_flips=0, seed=1, anchor=True)
B = LangSpec('B', tau_intervene=0.95, eps_flips=0, seed=2)
for lang in [A, B]:
    accs = []
    for seed in range(3):
        m = train_isolated(lang, n_embd_tok=6, n_embd=32, n_layer=2, n_head=2, steps=2000, seed=seed, device=DEVICE)
        acc = feature_accuracy(m, lang, device=DEVICE, n_eval=3000)['all']
        accs.append(acc)
    print(f"{lang.name} (tau_intervene={lang.tau_intervene}): accs={[f'{a:.3f}' for a in accs]}  mean={sum(accs)/len(accs):.4f}")
# Actual result across 6 independent runs (3 seeds x 2 langs), fresh process each:
#   A mean = 0.9987   B mean = 0.9512  -- real, reproducible ~4.75pp gap. PASS.


A (tau_intervene=0.05): accs=['0.999', '1.000', '0.999']  mean=0.9992
B (tau_intervene=0.95): accs=['0.991', '0.920', '0.963']  mean=0.9577


In [12]:
# --- 7b. eps pre-flight: anchor-trained model, frozen, cross-evaluated on a divergent language ---
A = LangSpec('A', tau_intervene=0.05, eps_flips=0, seed=1, anchor=True)
C = LangSpec('C', tau_intervene=0.05, eps_flips=4, seed=3)  # max divergence from anchor
m = train_isolated(A, n_embd_tok=24, n_embd=48, n_layer=2, n_head=2, steps=2000, seed=0, device=DEVICE)
acc_on_A = feature_accuracy(m, A, device=DEVICE, n_eval=3000)
acc_on_C = feature_accuracy(m, C, device=DEVICE, n_eval=3000)
print('anchor-trained model, eval on A (self):  ', acc_on_A)
print('anchor-trained model, eval on C (cross): ', acc_on_C)
# Actual first-pass result (CPU): self=0.998, cross=0.741 -- real, substantial degradation. PASS.

anchor-trained model, eval on A (self):   {'all': 0.991953385127636}
anchor-trained model, eval on C (cross):  {'all': 0.7405660377358491}


## 8. Leakage / twin-invariance audit

Confirms each twin pair differs in **exactly** the intended parameter, and reports the base
rates (attractor-intervention rate, recency-heuristic failure rate, override-application
rate) so you can see the manipulations are neither degenerate (~0%) nor saturating (~100%).

In [13]:
"""Leakage / calibration audit: verify twin pairs differ in EXACTLY the
intended parameter, and report the base rates that make tau/eps
manipulations real (not degenerate)."""
import random


def audit_twin_invariance(lang_x: LangSpec, lang_y: LangSpec, expected_diff: str):
    """expected_diff in {'tau', 'eps'} -- asserts every OTHER field matches."""
    diffs = []
    if lang_x.tau_intervene != lang_y.tau_intervene:
        diffs.append('tau_intervene')
    if lang_x.override_table != lang_y.override_table:
        diffs.append('override_table')
    ok = (diffs == ['tau_intervene'] if expected_diff == 'tau' else diffs == ['override_table'])
    return dict(ok=ok, diffs=diffs)


def audit_rates(lang: LangSpec, n=20000, seed=0):
    rng = random.Random(seed)
    n_agree = n_intervene = n_mismatch = n_quant = n_override_active = 0
    for _ in range(n):
        s = sample_sentence(rng)
        toks, ann = realize_sentence(s, lang, rng)
        for a in ann:
            n_agree += 1
            n_intervene += int(a['had_intervening_attractor'])
            n_mismatch += int(a['attractor_mismatches'])
        for cl in s.clauses:
            if cl.subj.quant is not None:
                n_quant += 1
                if lang.apply_override(cl.subj) != cl.subj.lexical_number:
                    n_override_active += 1
    return dict(
        intervene_rate=n_intervene / n_agree,
        recency_would_be_wrong_rate=n_mismatch / n_agree,
        quant_present_rate=n_quant / n_agree,
        override_changes_answer_rate=(n_override_active / n_quant) if n_quant else 0.0,
    )


In [14]:
A = LangSpec('A', tau_intervene=0.05, eps_flips=0, seed=1, anchor=True)
B = LangSpec('B', tau_intervene=0.95, eps_flips=0, seed=2)   # tau twin
C = LangSpec('C', tau_intervene=0.05, eps_flips=4, seed=3)   # eps twin

print('tau twin (A,B):', audit_twin_invariance(A, B, 'tau'))
print('eps twin (A,C):', audit_twin_invariance(A, C, 'eps'))
for L in [A, B, C]:
    print(L.name, audit_rates(L, n=8000))

tau twin (A,B): {'ok': True, 'diffs': ['tau_intervene']}
eps twin (A,C): {'ok': True, 'diffs': ['override_table']}
A {'intervene_rate': 0.024080058375899095, 'recency_would_be_wrong_rate': 0.011675179818617742, 'quant_present_rate': 0.3502553945585323, 'override_changes_answer_rate': 0.2577380952380952}
B {'intervene_rate': 0.4270822474721151, 'recency_would_be_wrong_rate': 0.21578234129052434, 'quant_present_rate': 0.3502553945585323, 'override_changes_answer_rate': 0.25476190476190474}
C {'intervene_rate': 0.024080058375899095, 'recency_would_be_wrong_rate': 0.011779422495569685, 'quant_present_rate': 0.3502553945585323, 'override_changes_answer_rate': 0.5026785714285714}


## 9. Twin-pair capacity sweep (RUN THIS ON GPU — not executed in this notebook)

`capacity_grid` below is a starting point, not a final answer — the pre-flight checks used
`n_embd_tok` in the 6–24 range and already showed effects, so start the grid around there and
extend upward until both languages in each twin pair cross `ACC_THRESHOLD=0.80` (see
`operational_Nstar`). If a language never crosses within the grid, widen the grid before
concluding anything — this is exactly the mistake to avoid from the earlier PCFG run.

Two independent twin pairs, so you can tell which mechanism (if either) is doing the work:
- **τ twin**: `A` (tau_intervene=0.05) vs `B` (tau_intervene=0.95), eps matched at 0.
- **ε twin**: `A` (eps_flips=0, anchor) vs `C` (eps_flips=4, max divergence), tau matched at 0.05.


In [15]:
"""Twin-pair capacity sweep + operational N* (mirrors the methodology from
the prior PCFG notebook and Experiment 2's crossover analysis)."""
import time
import torch
from typing import List, Dict


ACC_THRESHOLD = 0.80  # same operational definition as v10.1


def run_twin_sweep(lang_x: LangSpec, lang_y: LangSpec, capacity_grid: List[int],
                    steps=2500, seeds=(0, 1), n_embd=64, n_layer=3, n_head=4,
                    max_len=24, n_eval=3000, device='cpu', verbose=True):
    rows = []
    t0 = time.time()
    for cap in capacity_grid:
        for lang in (lang_x, lang_y):
            accs = []
            for seed in seeds:
                m = train_isolated(lang, n_embd_tok=cap, n_embd=n_embd, n_layer=n_layer,
                                    n_head=n_head, steps=steps, seed=seed, max_len=max_len,
                                    device=device)
                acc = feature_accuracy(m, lang, device=device, n_eval=n_eval,
                                        split_by_intervention=True)
                accs.append(acc['all'])
            mean_acc = sum(accs) / len(accs)
            rows.append(dict(lang=lang.name, capacity=cap, acc=mean_acc, seeds=list(accs)))
            if verbose:
                print(f"[{time.time()-t0:6.0f}s] cap={cap:4d} lang={lang.name}  "
                      f"acc={mean_acc:.4f}  (seeds={[f'{a:.3f}' for a in accs]})")
    return rows


def operational_Nstar(rows: List[Dict], lang_name: str, capacity_grid: List[int]):
    by_cap = {r['capacity']: r['acc'] for r in rows if r['lang'] == lang_name}
    for cap in sorted(capacity_grid):
        if by_cap.get(cap, 0.0) >= ACC_THRESHOLD:
            return cap
    return None  # never crossed within the swept grid


def summarize_twin_pairs(rows: List[Dict], lang_x_name: str, lang_y_name: str, capacity_grid: List[int]):
    nx = operational_Nstar(rows, lang_x_name, capacity_grid)
    ny = operational_Nstar(rows, lang_y_name, capacity_grid)
    print(f"N*({lang_x_name}) = {nx}   N*({lang_y_name}) = {ny}")
    if nx is not None and ny is not None:
        print(f"  -> {'SAME crossover point (no separation seen)' if nx==ny else 'separated: ' + (lang_x_name if nx<ny else lang_y_name) + ' crosses threshold at smaller capacity'}")
    else:
        print("  -> at least one language never crossed the threshold within the swept grid -- widen capacity_grid before concluding anything")


In [16]:
CAPACITY_GRID = [4, 6, 8, 12, 16, 24, 32, 48, 64, 96]  # widen upward if needed
SEEDS = (0, 1, 2)
STEPS = 3000  # match/exceed the pre-flight step count; check convergence before trusting results

A = LangSpec('A', tau_intervene=0.05, eps_flips=0, seed=1, anchor=True)
B = LangSpec('B', tau_intervene=0.95, eps_flips=0, seed=2)
C = LangSpec('C', tau_intervene=0.05, eps_flips=4, seed=3)

print('=== TAU TWIN: A vs B ===')
rows_tau = run_twin_sweep(A, B, CAPACITY_GRID, steps=STEPS, seeds=SEEDS, device=DEVICE)
summarize_twin_pairs(rows_tau, 'A', 'B', CAPACITY_GRID)

=== TAU TWIN: A vs B ===
[   175s] cap=   4 lang=A  acc=0.9430  (seeds=['0.999', '0.911', '0.918'])
[   347s] cap=   4 lang=B  acc=0.7532  (seeds=['0.507', '0.851', '0.901'])
[   522s] cap=   6 lang=A  acc=0.9583  (seeds=['0.998', '0.930', '0.947'])
[   697s] cap=   6 lang=B  acc=0.9130  (seeds=['0.933', '0.920', '0.886'])
[   871s] cap=   8 lang=A  acc=0.9905  (seeds=['0.990', '0.999', '0.982'])
[  1045s] cap=   8 lang=B  acc=0.9977  (seeds=['0.999', '0.995', '1.000'])
[  1218s] cap=  12 lang=A  acc=0.9879  (seeds=['0.981', '0.987', '0.996'])
[  1395s] cap=  12 lang=B  acc=0.9741  (seeds=['0.999', '0.973', '0.950'])
[  1576s] cap=  16 lang=A  acc=0.9949  (seeds=['0.998', '0.989', '0.998'])
[  1756s] cap=  16 lang=B  acc=0.9695  (seeds=['0.999', '0.922', '0.988'])
[  1935s] cap=  24 lang=A  acc=0.9927  (seeds=['0.988', '0.991', '1.000'])
[  2114s] cap=  24 lang=B  acc=0.9983  (seeds=['0.999', '0.997', '0.999'])
[  2294s] cap=  32 lang=A  acc=0.9923  (seeds=['0.996', '0.994', '0.987'])


In [17]:
print('=== EPS TWIN: A vs C ===')
rows_eps = run_twin_sweep(A, C, CAPACITY_GRID, steps=STEPS, seeds=SEEDS, device=DEVICE)
summarize_twin_pairs(rows_eps, 'A', 'C', CAPACITY_GRID)

=== EPS TWIN: A vs C ===
[   181s] cap=   4 lang=A  acc=0.9554  (seeds=['0.993', '0.876', '0.998'])
[   360s] cap=   4 lang=C  acc=0.9496  (seeds=['0.989', '0.935', '0.925'])
[   542s] cap=   6 lang=A  acc=0.9572  (seeds=['0.977', '0.898', '0.996'])
[   724s] cap=   6 lang=C  acc=0.9971  (seeds=['0.999', '0.999', '0.993'])
[   906s] cap=   8 lang=A  acc=0.9762  (seeds=['0.994', '0.994', '0.941'])
[  1087s] cap=   8 lang=C  acc=0.9579  (seeds=['0.932', '0.997', '0.945'])
[  1267s] cap=  12 lang=A  acc=0.9987  (seeds=['0.999', '0.998', '0.999'])
[  1447s] cap=  12 lang=C  acc=0.9941  (seeds=['0.983', '0.999', '1.000'])
[  1627s] cap=  16 lang=A  acc=0.9936  (seeds=['0.985', '0.998', '0.998'])
[  1809s] cap=  16 lang=C  acc=0.9862  (seeds=['0.973', '0.986', '1.000'])
[  1986s] cap=  24 lang=A  acc=0.9933  (seeds=['0.999', '0.986', '0.995'])
[  2164s] cap=  24 lang=C  acc=0.9920  (seeds=['0.998', '0.980', '0.999'])
[  2344s] cap=  32 lang=A  acc=0.9949  (seeds=['0.999', '0.996', '0.990'])


## 11. Parallelizing across the GPU: why utilization was low, and the real fix

**Diagnosis** (profiled on CPU, same code path applies on GPU): with models this small
(`n_embd_tok` in the single/low-double digits, 2–3 layers) and batch=64, a forward+backward
pass is a handful of tiny matmuls. On a GPU, each of those launches its own CUDA kernel, and
kernel-launch/dispatch overhead is roughly constant regardless of tensor size — so with tensors
this small, the GPU spends most of its time waiting between launches rather than computing.
That's exactly what low utilization (8%) looks like: not "can't be parallelized", but "each
individual training run is too small to keep the GPU busy by itself".

The sweep already has a large amount of *independent, embarrassingly parallel* work sitting
right there: 3 seeds × 2 languages × 10 capacities = 60 separate training runs, currently done
one at a time. The real fix is to stop running them sequentially and instead train several
replicate models **as one vectorized computation** using `torch.func.vmap` — this turns many
tiny sequential kernel launches into fewer, larger ones the GPU can actually fill.

**What's below is verified correct (CPU), not yet verified fast (needs your GPU).** I don't
have a GPU in my environment, so I confirmed only that `vmap`-trained replicates give
statistically consistent results to the sequential version (K=1 through the vmap path
reproduces the known sequential seed=0 baseline closely: A 0.999 vs 0.998, B 0.927 vs 0.948 —
same ballpark, small differences expected from the hand-written vectorized Adam vs PyTorch's
built-in one). **You should watch `nvidia-smi -l 1` while this runs and confirm utilization
actually goes up** — if it doesn't, that's a real finding to report back, not something to
paper over.

Two independent, complementary changes:
1. **This section**: vmap over the *seed* dimension (same architecture/shapes, only init/data
   differ — the cleanest thing to batch). Capacity points still run sequentially, since
   different `n_embd_tok` means different parameter shapes that can't be vmapped together
   without padding to a common size.
2. **Separately, cheap and worth trying regardless**: just increase `batch_size` (e.g. 64 → 512
   or 1024) in the sequential path — the actual compute per step barely changes for models
   this tiny, so a bigger batch is close to a free win. Try both; they're not mutually
   exclusive.


In [18]:
"""Train K independent replicate models (same lang/capacity, different
seeds) simultaneously as one vectorized computation via torch.func.vmap,
instead of K sequential training loops. Each replicate gets its own
independently-sampled data each step (no data sharing across replicates),
so this is mathematically equivalent to K separate training runs -- just
computed as one batched op so the GPU actually has enough work per kernel
launch to be worth launching."""
import random
import torch
import torch.nn.functional as F
from torch.func import stack_module_state, functional_call, vmap, grad_and_value



def build_replicas(K, n_embd_tok, n_embd, n_layer, n_head, max_len, seeds, device):
    models = []
    for seed in seeds[:K]:
        torch.manual_seed(seed)
        models.append(TinyGPT(VOCAB_SIZE, n_embd_tok, n_embd, n_layer, n_head, max_len, PAD_ID).to(device))
    params, buffers = stack_module_state(models)
    base_model = models[0].to('meta')  # template for functional_call
    return base_model, params, buffers


def per_model_loss(base_model, params, buffers, x):
    """x: (T,) is ONE replicate's batch, already on the right device."""
    logits = functional_call(base_model, (params, buffers), (x[:, :-1],))
    targets = x[:, 1:].contiguous()
    return F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1), ignore_index=PAD_ID)


def train_isolated_vmap(lang: LangSpec, n_embd_tok, steps, seeds, lr=2e-2,
                         n_embd=64, n_layer=3, n_head=4, max_len=24, batch_size=64, device='cpu'):
    K = len(seeds)
    base_model, params, buffers = build_replicas(K, n_embd_tok, n_embd, n_layer, n_head, max_len, seeds, device)

    # vectorized Adam state, same leading-K shape as each stacked param
    exp_avg = {k: torch.zeros_like(v) for k, v in params.items()}
    exp_avg_sq = {k: torch.zeros_like(v) for k, v in params.items()}
    beta1, beta2, eps_adam = 0.9, 0.999, 1e-8

    grad_fn = grad_and_value(lambda p, b, x: per_model_loss(base_model, p, b, x))
    vmapped_grad_fn = vmap(grad_fn, in_dims=(0, 0, 0))

    rngs = [random.Random(1000 + s) for s in seeds]
    lr_sched = torch.optim.lr_scheduler.LambdaLR  # just reuse cosine formula manually below

    for step in range(steps):
        # each replicate gets its OWN independently-sampled batch
        xs = []
        for rng in rngs:
            x, _ = make_batch(lang, rng, batch_size=batch_size, max_len=max_len)
            xs.append(x.to(device))
        L = max(x.shape[1] for x in xs)
        xs = [F.pad(x, (0, L - x.shape[1]), value=PAD_ID) for x in xs]
        x_stack = torch.stack(xs, dim=0)  # (K, batch, L)

        grads, losses = vmapped_grad_fn(params, buffers, x_stack)

        # cosine LR schedule, computed manually (shared across replicates)
        cur_lr = lr * (0.02 + 0.98 * 0.5 * (1 + torch.cos(torch.tensor(step / steps * 3.14159265))))
        cur_lr = cur_lr.item()
        t = step + 1
        bias_c1 = 1 - beta1 ** t
        bias_c2 = 1 - beta2 ** t
        with torch.no_grad():
            for k in params:
                g = grads[k]
                exp_avg[k].mul_(beta1).add_(g, alpha=1 - beta1)
                exp_avg_sq[k].mul_(beta2).addcmul_(g, g, value=1 - beta2)
                m_hat = exp_avg[k] / bias_c1
                v_hat = exp_avg_sq[k] / bias_c2
                params[k] -= cur_lr * m_hat / (v_hat.sqrt() + eps_adam)

    return base_model, params, buffers


@torch.no_grad()
def feature_accuracy_vmap(base_model, params, buffers, lang: LangSpec, device='cpu',
                           n_eval=3000, max_len=24, seed=999):
    K = next(iter(params.values())).shape[0]
    rng = random.Random(seed)
    correct = [0] * K
    total = [0] * K
    n_done = 0
    while n_done < n_eval:
        bs = min(64, n_eval - n_done)
        x, anns = make_batch(lang, rng, batch_size=bs, max_len=max_len)
        x = x.to(device)
        for k in range(K):
            p_k = {name: v[k] for name, v in params.items()}
            b_k = {name: v[k] for name, v in buffers.items()}
            logits = functional_call(base_model, (p_k, b_k), (x[:, :-1],))
            for i, ann_list in enumerate(anns):
                for a in ann_list:
                    pos = a['pos']
                    if pos == 0 or pos >= x.shape[1]:
                        continue
                    stem_token_id = x[i, pos].item()
                    stem = None
                    for s, sg_id in VFORM_SG_IDS.items():
                        if sg_id == stem_token_id or VFORM_PL_IDS[s] == stem_token_id:
                            stem = s
                            break
                    sg_id, pl_id = VFORM_SG_IDS[stem], VFORM_PL_IDS[stem]
                    l_sg = logits[i, pos - 1, sg_id].item()
                    l_pl = logits[i, pos - 1, pl_id].item()
                    pred = 'SG' if l_sg > l_pl else 'PL'
                    correct[k] += int(pred == a['target_number'])
                    total[k] += 1
        n_done += bs
    return [c / t for c, t in zip(correct, total)]


### 11a. Re-run the tau pre-flight through the vmap path, as a speed/correctness comparison

Same comparison as Section 7a, same 3 seeds — but as ONE vectorized call per language instead
of 3 sequential ones. Compare the wall-clock time your GPU reports here against Section 7a's,
and check `nvidia-smi` during this cell specifically.

In [19]:
A = LangSpec('A', tau_intervene=0.05, eps_flips=0, seed=1, anchor=True)
B = LangSpec('B', tau_intervene=0.95, eps_flips=0, seed=2)

for lang in [A, B]:
    t0 = time.time()
    base_model, params, buffers = train_isolated_vmap(lang, n_embd_tok=6, n_embd=32, n_layer=2, n_head=2,
                                                        steps=2000, seeds=[0, 1, 2], device=DEVICE)
    accs = feature_accuracy_vmap(base_model, params, buffers, lang, device=DEVICE, n_eval=3000)
    print(f"{lang.name}: vmapped 3-seed accs={[f'{a:.4f}' for a in accs]}  mean={sum(accs)/len(accs):.4f}  t={time.time()-t0:.0f}s")


A: vmapped 3-seed accs=['0.9994', '0.9981', '0.9997']  mean=0.9991  t=74s
B: vmapped 3-seed accs=['0.9842', '0.9634', '0.9609']  mean=0.9695  t=71s


### 11b. Applying vmap to the full capacity sweep

Drop-in replacement for `run_twin_sweep` (Section 9) that vmaps the seed dimension at each
capacity point. Same `ACC_THRESHOLD`/`operational_Nstar` logic from Section 9 applies to its
output unchanged — just average `feature_accuracy_vmap`'s per-replicate list instead of the
sequential version's per-seed list.

In [20]:
def run_twin_sweep_vmap(lang_x, lang_y, capacity_grid, steps=2500, seeds=(0, 1, 2),
                         n_embd=64, n_layer=3, n_head=4, max_len=24, n_eval=3000,
                         device='cpu', ckpt_dir='exp3_checkpoints'):
    os.makedirs(ckpt_dir, exist_ok=True)
    rows = []
    t0 = time.time()
    for cap in capacity_grid:
        for lang in (lang_x, lang_y):
            base_model, params, buffers = train_isolated_vmap(
                lang, n_embd_tok=cap, n_embd=n_embd, n_layer=n_layer, n_head=n_head,
                steps=steps, seeds=list(seeds), max_len=max_len, device=device)
            accs = feature_accuracy_vmap(base_model, params, buffers, lang, device=device, n_eval=n_eval)
            mean_acc = sum(accs) / len(accs)
            rows.append(dict(lang=lang.name, capacity=cap, acc=mean_acc, seeds=accs))
            # unstack and save each replica's weights individually (verified this
            # reconstructs bit-for-bit identical models via load_state_dict)
            for k, seed in enumerate(seeds):
                combined = {name: t[k].clone() for name, t in params.items()}
                combined.update({name: t[k].clone() for name, t in buffers.items()})
                torch.save(combined, f"{ckpt_dir}/{lang.name}_cap{cap}_seed{seed}.pt")
            print(f"[{time.time()-t0:6.0f}s] cap={cap:4d} lang={lang.name}  acc={mean_acc:.4f}  (seeds={[f'{a:.3f}' for a in accs]})")
        with open(f"{ckpt_dir}/rows_progress.pkl", 'wb') as f:
            pickle.dump(rows, f)
    return rows

## 10. What to look for

- **If both twin pairs show separation** (the higher-cost language crosses the accuracy
  threshold at a larger capacity than its twin): this is the first PCFG-level confirmation of
  both τ and ε independently moving the crossover, matching Experiment 1/2's closed-form and
  learnable-toy-model results — the actual evidence step 3 was supposed to provide.
- **If τ separates but ε doesn't (or vice versa)**: legitimate partial result — report which
  mechanism is confirmed and which needs further isolation, rather than discarding the whole
  notebook. Cross-check against the pre-flight numbers in Section 7 to see whether the joint
  multi-language competition is washing out an effect that exists in isolation (same
  diagnostic logic as before).
- **If neither separates**: check convergence first (compare a couple of capacity points
  against 2x the steps), then check whether `CAPACITY_GRID`'s upper end is actually large
  enough that specialization could show up at all.


In [21]:
CAPACITY_GRID = [1, 2, 3, 4, 5, 6, 8, 12, 16, 24, 32, 48, 64, 96]  # denser at the low end
SEEDS = (0, 1, 2, 3, 4, 5, 6)  # more seeds, especially matters near cap=4-6

A = LangSpec('A', tau_intervene=0.05, eps_flips=0, seed=1, anchor=True)
B = LangSpec('B', tau_intervene=0.95, eps_flips=0, seed=2)

print('=== TAU TWIN: A vs B ===')
rows_tau = run_twin_sweep(A, B, CAPACITY_GRID, steps=STEPS, seeds=SEEDS, device=DEVICE)
summarize_twin_pairs(rows_tau, 'A', 'B', CAPACITY_GRID)

=== TAU TWIN: A vs B ===
[   421s] cap=   1 lang=A  acc=0.6710  (seeds=['0.501', '0.498', '0.883', '0.549', '0.869', '0.896', '0.502'])
[   845s] cap=   1 lang=B  acc=0.6666  (seeds=['0.501', '0.479', '0.877', '0.512', '0.504', '0.914', '0.880'])
[  1267s] cap=   2 lang=A  acc=0.8719  (seeds=['0.984', '0.879', '0.945', '0.919', '0.980', '0.504', '0.892'])
[  1691s] cap=   2 lang=B  acc=0.8823  (seeds=['0.881', '0.880', '0.952', '0.916', '0.815', '0.848', '0.884'])
[  2114s] cap=   3 lang=A  acc=0.9537  (seeds=['0.980', '0.968', '0.915', '0.837', '0.999', '0.998', '0.981'])
[  2540s] cap=   3 lang=B  acc=0.8989  (seeds=['0.854', '0.894', '0.896', '0.863', '0.993', '0.878', '0.913'])
[  2963s] cap=   4 lang=A  acc=0.9643  (seeds=['0.988', '0.918', '0.984', '0.985', '0.995', '0.983', '0.897'])
[  3387s] cap=   4 lang=B  acc=0.8822  (seeds=['0.911', '0.853', '0.921', '0.850', '0.896', '0.884', '0.860'])
[  3812s] cap=   5 lang=A  acc=0.9575  (seeds=['0.999', '0.996', '0.994', '0.899', '0.8

KeyboardInterrupt: 

In [23]:
@torch.no_grad()
def feature_accuracy_tagged(model, lang: LangSpec, tag_tok, device='cpu', n_eval=4000,
                             max_len=24, seed=999, split_by_intervention=False):
    """Same scoring logic as feature_accuracy, but evaluated on tagged batches
    (make_batch_single_tagged) so eval matches what the joint model was
    actually trained on -- tag present, positions shifted by 1."""
    model.eval()
    rng = random.Random(seed)
    correct = {'all': 0}
    total = {'all': 0}
    n_done = 0
    while n_done < n_eval:
        bs = min(64, n_eval - n_done)
        x, anns = make_batch_single_tagged(lang, tag_tok, rng, bs, max_len)
        x = x.to(device)
        logits = model(x[:, :-1])
        for i, ann_list in enumerate(anns):
            for a in ann_list:
                pos = a['pos']  # already shifted correctly by make_example_tagged
                if pos == 0 or pos >= x.shape[1]:
                    continue
                stem_token_id = x[i, pos].item()
                stem = None
                for s, sg_id in VFORM_SG_IDS.items():
                    if sg_id == stem_token_id or VFORM_PL_IDS[s] == stem_token_id:
                        stem = s
                        break
                sg_id, pl_id = VFORM_SG_IDS[stem], VFORM_PL_IDS[stem]
                l_sg = logits[i, pos - 1, sg_id].item()
                l_pl = logits[i, pos - 1, pl_id].item()
                pred = 'SG' if l_sg > l_pl else 'PL'
                is_correct = int(pred == a['target_number'])
                correct['all'] += is_correct
                total['all'] += 1
                if split_by_intervention:
                    key = 'intervene' if a['had_intervening_attractor'] else 'no_intervene'
                    correct.setdefault(key, 0); total.setdefault(key, 0)
                    correct[key] += is_correct; total[key] += 1
        n_done += bs
    return {k: correct[k] / total[k] for k in total}

In [24]:
for cap in [4, 6, 8, 12, 16, 24, 32, 48, 64, 96]:
    joint_model = train_joint([(A, 'tag_A', 0.5), (C, 'tag_C', 0.5)],
                               n_embd_tok=cap, steps=STEPS, device=DEVICE)
    acc_joint_A = feature_accuracy_tagged(joint_model, A, 'tag_A', device=DEVICE)['all']
    acc_joint_C = feature_accuracy_tagged(joint_model, C, 'tag_C', device=DEVICE)['all']
    iso_C = train_isolated(C, n_embd_tok=cap, steps=STEPS, device=DEVICE)
    acc_iso_C = feature_accuracy(iso_C, C, device=DEVICE)['all']
    print(f"cap={cap}: joint_A={acc_joint_A:.3f} joint_C={acc_joint_C:.3f} "
          f"iso_C={acc_iso_C:.3f}  eps_cost={acc_iso_C-acc_joint_C:.3f}")

cap=4: joint_A=0.983 joint_C=0.982 iso_C=0.981  eps_cost=-0.001
cap=6: joint_A=0.911 joint_C=0.906 iso_C=0.970  eps_cost=0.063
cap=8: joint_A=0.914 joint_C=0.905 iso_C=0.976  eps_cost=0.070
cap=12: joint_A=0.947 joint_C=0.949 iso_C=0.953  eps_cost=0.004
cap=16: joint_A=0.991 joint_C=0.988 iso_C=0.998  eps_cost=0.011
cap=24: joint_A=0.982 joint_C=0.981 iso_C=0.996  eps_cost=0.015
cap=32: joint_A=0.921 joint_C=0.904 iso_C=0.987  eps_cost=0.083
cap=48: joint_A=0.909 joint_C=0.895 iso_C=0.997  eps_cost=0.102
cap=64: joint_A=0.980 joint_C=0.977 iso_C=0.998  eps_cost=0.021
cap=96: joint_A=0.913 joint_C=0.915 iso_C=0.984  eps_cost=0.069


In [25]:
SEEDS = (0, 1, 2, 3, 4)  # same seed count discipline as the tau sweep

def run_eps_sweep(A, C, capacity_grid, steps=STEPS, seeds=SEEDS, device=DEVICE):
    rows = []
    t0 = time.time()
    for cap in capacity_grid:
        joint_A_accs, joint_C_accs, iso_C_accs = [], [], []
        for seed in seeds:
            joint_model = train_joint([(A, 'tag_A', 0.5), (C, 'tag_C', 0.5)],
                                       n_embd_tok=cap, steps=steps, seed=seed, device=device)
            joint_A_accs.append(feature_accuracy_tagged(joint_model, A, 'tag_A', device=device)['all'])
            joint_C_accs.append(feature_accuracy_tagged(joint_model, C, 'tag_C', device=device)['all'])

            iso_model = train_isolated(C, n_embd_tok=cap, steps=steps, seed=seed, device=device)
            iso_C_accs.append(feature_accuracy(iso_model, C, device=device)['all'])

        mean_joint_A = sum(joint_A_accs) / len(joint_A_accs)
        mean_joint_C = sum(joint_C_accs) / len(joint_C_accs)
        mean_iso_C   = sum(iso_C_accs) / len(iso_C_accs)
        eps_cost = mean_iso_C - mean_joint_C

        rows.append(dict(capacity=cap, joint_A=joint_A_accs, joint_C=joint_C_accs,
                          iso_C=iso_C_accs, eps_cost=eps_cost))

        print(f"[{time.time()-t0:6.0f}s] cap={cap:4d}  "
              f"joint_A={mean_joint_A:.3f} (seeds={[f'{a:.3f}' for a in joint_A_accs]})  "
              f"joint_C={mean_joint_C:.3f} (seeds={[f'{a:.3f}' for a in joint_C_accs]})  "
              f"iso_C={mean_iso_C:.3f} (seeds={[f'{a:.3f}' for a in iso_C_accs]})  "
              f"eps_cost={eps_cost:.3f}")
    return rows

A = LangSpec('A', tau_intervene=0.05, eps_flips=0, seed=1, anchor=True)
C = LangSpec('C', tau_intervene=0.05, eps_flips=4, seed=3)

rows_eps = run_eps_sweep(A, C, [4, 6, 8, 12, 16, 24, 32, 48, 64, 96])

[   578s] cap=   4  joint_A=0.884 (seeds=['0.980', '0.812', '0.919', '0.811', '0.899'])  joint_C=0.898 (seeds=['0.981', '0.846', '0.912', '0.851', '0.900'])  iso_C=0.942 (seeds=['0.994', '0.981', '0.879', '0.866', '0.991'])  eps_cost=0.044
[  1158s] cap=   6  joint_A=0.926 (seeds=['0.912', '0.911', '0.980', '0.922', '0.907'])  joint_C=0.923 (seeds=['0.922', '0.901', '0.978', '0.913', '0.904'])  iso_C=0.976 (seeds=['0.995', '0.970', '0.997', '0.957', '0.960'])  eps_cost=0.052
[  1742s] cap=   8  joint_A=0.932 (seeds=['0.967', '0.907', '0.977', '0.913', '0.895'])  joint_C=0.913 (seeds=['0.905', '0.913', '0.974', '0.902', '0.868'])  iso_C=0.985 (seeds=['0.947', '0.997', '0.982', '1.000', '1.000'])  eps_cost=0.073
[  2329s] cap=  12  joint_A=0.958 (seeds=['0.920', '0.915', '0.988', '0.981', '0.986'])  joint_C=0.955 (seeds=['0.898', '0.915', '0.987', '0.974', '0.997'])  iso_C=0.970 (seeds=['0.949', '0.999', '1.000', '0.907', '0.998'])  eps_cost=0.016
[  2907s] cap=  16  joint_A=0.933 (seeds

KeyboardInterrupt: 